# ECoG Decoding Algorithm Prototype

This notebook is a compact prototype for ECoG-based movement or stimulus decoding. It focuses on the core algorithmic path: loading matrix data, removing bad channels, filtering signals, extracting frequency-domain features, and training regression models.

The original non-public data paths have been replaced with generic placeholders. The notebook is intended to show the technical workflow rather than serve as a fully reproducible public dataset release.

In [ ]:
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft, ifft
from scipy.stats import pearsonr
from scipy import signal as sig
from scipy.signal import ellip,lfilter,filtfilt,find_peaks,firwin,kaiserord, decimate, periodogram
from copy import deepcopy
from scipy import stats
import scipy
import warnings
warnings.filterwarnings('ignore')
from scipy.fft import rfft, rfftfreq
from scipy import signal
from scipy.integrate import simps
from sklearn. decomposition import PCA
from scipy.interpolate import CubicSpline, splrep, splev
from sklearn.linear_model import LinearRegression
from scipy.io import savemat
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import scale
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler #standard scaler worked best
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from keras.layers import Dense
from tensorflow.keras.layers import ReLU, LeakyReLU, BatchNormalization, Flatten, Conv2D, MaxPooling2D, Conv1D
import joblib
from scipy.stats import entropy
from scipy.signal import welch


In [ ]:
from tqdm import tqdm

## Data Loading

The workflow starts from MATLAB-format ECoG and target-variable arrays. The signal matrix is organized by subject, channel, and time, while the target matrix contains the continuous variables used for decoding.

In [ ]:
# Load .mat files
path_to_files = 'data/ecog_decoding_project/'  # Local demo data directory.

train_data = scipy.io.loadmat(path_to_files + 'raw_training_data.mat')
leaderboard_data = scipy.io.loadmat(path_to_files + 'leaderboard_data.mat')

train_dg = train_data['train_dg']
train_ecog = train_data['train_ecog']
test_data = leaderboard_data['leaderboard_ecog']

fs = 1000 #Hz

## Bad-Channel Handling

Known noisy channels are removed before filtering and feature extraction. This step mirrors a common biosignal-processing practice: excluding channels with unstable variance, poor contact, or unusable signal quality before model training.

In [ ]:
#subject 1
#bad channels are : 54
train_ecog[0][0] = np.delete(train_ecog[0][0], 54, 1)
test_data[0][0] = np.delete(test_data[0][0], 54, 1)

#subject 2
#bad channels are : 20 and 37
train_ecog[1][0] = np.delete(train_ecog[1][0], [20,37], 1)
test_data[1][0] = np.delete(test_data[1][0], [20,37], 1)

#subject 3
# bad channels are : None

## Train/Test Split

Each subject is split into training and testing segments before feature extraction outputs are passed to the model. Keeping this split explicit helps prevent leakage from later preprocessing or windowing steps.

In [ ]:
#TRAIN SET
#train labels
train_dg1 = train_dg[0][0]
train_dg2 = train_dg[1][0]
train_dg3 = train_dg[2][0]

#train data
train_ecog1 = train_ecog[0][0]
train_ecog2 = train_ecog[1][0]
train_ecog3 = train_ecog[2][0]

#TEST SET
#test data
test_ecog1 = test_data[0][0]
test_ecog2 = test_data[1][0]
test_ecog3 = test_data[2][0]

## Preprocessing and Feature Construction

The preprocessing block applies filtering, windowing, and spectral feature calculation. Features are derived from physiologically meaningful frequency bands and assembled into the design matrices used by the decoding models.

In [ ]:
def filter_data_2(raw_eeg, fs=1000):
  nyq=fs/2
  ripple_db=60.0 #ripple size for cutoff
  width=5.0/nyq #transition wid
  N, beta = kaiserord(ripple_db, width) #kaiser parameters
  taps_default=firwin(N,[4, 200],window=('kaiser',beta),pass_zero='bandpass',fs=1000)
  filtered_2=filtfilt(taps_default,1.0,raw_eeg)
  return filtered_2

def filternotch(x,fs=1000):
  b1, a1 = sig.iirnotch(60, 20, fs)
  b2, a2 = sig.iirnotch(120, 20, fs)
  b3, a3 = sig.iirnotch(180, 20, fs)
  output1=sig.filtfilt(b1,a1,x)
  output2=sig.filtfilt(b2,a2,output1)
  output3=sig.filtfilt(b3,a3,output2)
  return output3

In [ ]:
def NumWins(x,fs,winLen,winDisp):
  M =np.floor(((len(x)/fs) - winLen + winDisp)/ winDisp)
  return int(M)

def NumWins_no_overlap(x,winLen):
  M =np.divide((len(x)/fs), winLen)
  return int(M)

#feature 1
def Avg_voltage(x):
  avg_vol = np.mean(x)
  return avg_vol

#feature 2
def LL(x):
  ll_x = np.sum(np.absolute(np.ediff1d(x)))
  return ll_x

#feature 3
def Energy(x):
  E_x = np.sum(np.square(x))
  return E_x

#feature 4 ,5, 6, 7, 8
def Avg_Freq(x, fi, ff):
  #Convert to frequency domain:
  freq_sig = rfft(x)
  N = len(x)
  tdomain=rfftfreq(N,1/fs)
  indices = np.where((tdomain>=fi) & (tdomain<=ff))[0]
  return np.mean(np.abs(freq_sig[indices]))

def peak_to_peak(x):
    return np.ptp(x)

def zero_crossing_rate(x):
    return ((x[:-1] * x[1:]) < 0).sum()

def variance(x):
    return np.var(x)

def spectral_entropy(x, fs):
    f, Pxx = welch(x, fs=fs)
    Pxx = Pxx / np.sum(Pxx)  # Normalize the power spectrum
    return entropy(Pxx)

def spectral_centroid(x, fs):
    f, Pxx = welch(x, fs=fs)
    centroid = np.sum(f * Pxx) / np.sum(Pxx)
    return centroid

In [ ]:
def get_features(filtered_window, fs=1000):

  channels = np.shape(filtered_window)[1]
  features = np.empty([channels, 13])
  for ch in range(channels):
    feat1 = Avg_voltage(filtered_window[:,ch])
    feat2 = LL(filtered_window[:,ch])
    feat3 = Energy(filtered_window[:,ch])
    feat4 = Avg_Freq(filtered_window[:,ch], 5, 15)
    feat5 = Avg_Freq(filtered_window[:,ch], 20, 25)
    feat6 = Avg_Freq(filtered_window[:,ch], 75, 115)
    feat7 = Avg_Freq(filtered_window[:,ch], 125, 160)
    feat8 = Avg_Freq(filtered_window[:,ch], 160, 175)
    feat9 = peak_to_peak(filtered_window[:,ch])
    feat10 = zero_crossing_rate(filtered_window[:,ch])
    feat11 = variance(filtered_window[:,ch])
    feat12 = spectral_entropy(filtered_window[:,ch], 1000)
    feat13 = spectral_centroid(filtered_window[:,ch], 1000)

    features[ch,:] = [feat1, feat2, feat3, feat4, feat5, feat6, feat7, feat8, feat9, feat10,feat11,feat12, feat13]

  features = np.reshape(features,(channels*13))

  return features

In [ ]:
def apply_medfilt_to_predictions(pred):
    """
    Applies median filter to a prediction array.

    Args:
    pred (np.array): Prediction array with shape (100000, 5).

    Returns:
    np.array: The filtered predictions with shape (100000, 5).
    """
    # Initialize the final predictions array
    final_pre = np.zeros(pred.shape)  # Assuming pred is already (100000, 5)

    # Apply median filter to each finger's predictions
    for finger in range(5):
        final_pre[:, finger] = sig.medfilt(pred[:, finger], kernel_size=601)

    return final_pre

In [ ]:
def get_windowed_feats(raw_ecog, fs, window_length, window_overlap):

  filtered_eeg = np.empty(np.shape(raw_ecog))

  for ch in range(np.shape(raw_ecog)[1]):
    filtered_eeg[:,ch] = filternotch(filter_data_2(raw_ecog[:,ch]))

  M = NumWins(filtered_eeg, fs, window_length, window_overlap)
  xLen = len(filtered_eeg)

  L = window_length
  d = window_overlap

  feature_vector = []
  for i in tqdm(range(int(M))):
    feature_values = get_features(filtered_eeg[int(xLen - ((L+i*d)*fs)) : int(xLen-(i*d*fs)), :])
    feature_vector.append(feature_values)
  feature_vector = np.array(feature_vector)
  feature_vector=feature_vector[::-1,:]

  return feature_vector

In [ ]:
def get_windowed_dg(raw_dg, fs, window_length, window_overlap):

  M = NumWins(raw_dg, fs, window_length, window_overlap)
  xLen = len(raw_dg)
  L = window_length
  d = window_overlap

  downsampled_dg = np.empty((M, 5))
  for i in tqdm(range(int(M))):
    for ch in range(5):
      #feat = np.mean(raw_dg[int(xLen - ((L+i*d)*fs)) : int(xLen-(i*d*fs)), ch])             #option1
      feat = np.max(raw_dg[int(xLen - ((L+i*d)*fs)) : int(xLen-(i*d*fs)), ch])               #option2 - BEST FINAL
      #feat = signal.decimate(raw_dg[int(xLen - ((L+i*d)*fs)) : int(xLen-(i*d*fs)), ch], M)  #option3
      downsampled_dg[i,ch] = feat

  downsampled_dg=downsampled_dg[::-1,:]
  return downsampled_dg

In [ ]:
def create_R_matrix(features, N_wind):

  M, ch = np.shape(features)
  feats2 = np.empty([M+N_wind-1, ch])

  feats2[N_wind-1:,:] = deepcopy(features)
  for i in range(0,N_wind-1):
    feats2[i, :] = deepcopy(features[i,:])

  R = np.empty([M, ch*N_wind + 1])
  R[:,0] = np.ones((M))

  for i in range(N_wind-1, np.shape(feats2)[0]):
    temp_arr = []
    for n in np.arange(0,N_wind)[::-1]:
      temp_arr = np.concatenate((temp_arr, feats2[i-(n),:]), axis=None)

    R[i-(N_wind-1), 1:] = temp_arr

  return R

## Subject-Level Feature Matrices

For each subject, the notebook builds paired train/test matrices for neural features and target variables. This keeps model fitting and evaluation separate across all subjects.

In [ ]:
#----------SUBJECT 1-------------
#X_test
X_test_1 = get_windowed_feats(test_ecog1, 1000, 0.1,0.05)
X_test_1 = np.nan_to_num(X_test_1)

R1_test = create_R_matrix(X_test_1, 5)

In [ ]:
#----------SUBJECT 2-------------
#X_test
X_test_2 = get_windowed_feats(test_ecog2, 1000, 0.1,0.05)
X_test_2 = np.nan_to_num(X_test_2)

R2_test = create_R_matrix(X_test_2, 5)

In [ ]:
#----------SUBJECT 3-------------
#X_test
X_test_3 = get_windowed_feats(test_ecog3, 1000, 0.1,0.05)
X_test_3 = np.nan_to_num(X_test_3)

R3_test = create_R_matrix(X_test_3, 5)

In [ ]:
#----------SUBJECT 1-------------

#X_train
X_train_1 = get_windowed_feats(train_ecog1, 1000, 0.1,0.05)
X_train_1 = np.nan_to_num(X_train_1)

#X_test
X_test_1 = get_windowed_feats(test_ecog1, 1000, 0.1,0.05)
X_test_1 = np.nan_to_num(X_test_1)

#y_train
y_train_1 = get_windowed_dg(train_dg1, 1000, 0.1, 0.05)

#R calc
R1_train = create_R_matrix(X_train_1, 5)


100%|██████████| 5998/5998 [00:00<00:00, 31214.87it/s]


In [ ]:
#----------SUBJECT 2-------------

#X_train
X_train_2 = get_windowed_feats(train_ecog2, 1000, 0.1,0.05)
X_train_2 = np.nan_to_num(X_train_2)

#X_test
X_test_2 = get_windowed_feats(test_ecog2, 1000, 0.1,0.05)
X_test_2 = np.nan_to_num(X_test_2)

#y_train
y_train_2 = get_windowed_dg(train_dg2, 1000, 0.1, 0.05)

#R calc
R2_train = create_R_matrix(X_train_2, 5)
R2_test = create_R_matrix(X_test_2, 5)

100%|██████████| 5998/5998 [00:00<00:00, 31948.63it/s]


In [ ]:
#----------SUBJECT 3-------------

#X_train
X_train_3 = get_windowed_feats(train_ecog3, 1000, 0.1,0.05)
X_train_3 = np.nan_to_num(X_train_3)

#X_test
X_test_3 = get_windowed_feats(test_ecog3, 1000, 0.1,0.05)
X_test_3 = np.nan_to_num(X_test_3)

#y_train
y_train_3 = get_windowed_dg(train_dg3, 1000, 0.1, 0.05)

#R calc
R3_train = create_R_matrix(X_train_3, 5)
R3_test = create_R_matrix(X_test_3, 5)

100%|██████████| 5998/5998 [00:00<00:00, 30693.12it/s]


## Neural Network Regression

A small neural network is used as a nonlinear decoder after the handcrafted feature pipeline. The model is intentionally lightweight so performance can be compared with simpler regression baselines.

### Subject 1

Subject-specific training and prediction are run independently to avoid mixing neural patterns across recording sessions.

In [ ]:
#-----------Subject 1-------------
#load the saved scaler
scaler_filename = "scaler1.save"
scaler1 = joblib.load(scaler_filename)

#transform the test data using the loaded scaler
R1_test = scaler1.transform(R1_test)

In [ ]:
from keras.models import load_model

#finger 1
#load the saved model
model11 = load_model('model11.h5')
#predictions
test_pred11 = model11.predict(R1_test)

#finger 2
#load the saved model
model12 = load_model('model12.h5')
#predictions
test_pred12 = model12.predict(R1_test)

#finger 3
#load the saved model
model13 = load_model('model13.h5')
#predictions
test_pred13 = model13.predict(R1_test)

#finger 4
#load the saved model
model14 = load_model('model14.h5')
#predictions
test_pred14 = model14.predict(R1_test)

#finger 5
#load the saved model
model15 = load_model('model15.h5')
#predictions
test_pred15 = model15.predict(R1_test)

93/93 [==============================] - 1s 6ms/step
[[-0.2729899 ]
 [-0.28206474]
 [-0.23728219]
 ...
 [-0.19115046]
 [-0.26027587]
 [-0.26773256]]


In [ ]:
#-----------Subject 1-------------

#scaling (normalizing) the data
scaler1 = StandardScaler()
R1_train = scaler1.fit_transform(R1_train)

scaler_filename = "scaler1.save"
joblib.dump(scaler1, scaler_filename)

R1_test = scaler1.transform(R1_test)

In [ ]:
#creating the neural network architecture
model = Sequential()
model.add(Dense(128, input_dim = R1_train.shape[1], activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(1, activation = 'linear'))

#choosing optimizer and performance metric
model.compile(optimizer = 'adam', loss = 'mean_absolute_error')

#early stopping
early_stop = EarlyStopping(monitor = 'val_loss', mode = 'min', verbose=1, patience = 25)

In [ ]:
#finger 1

#model.fit with early stopping
model.fit(R1_train, y_train_1[:,0], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model11.h5')

#predictions
test_pred11 = model.predict(R1_test)

Epoch 1/500
169/169 [==============================] - 4s 13ms/step - loss: 0.5313 - val_loss: 1.3240
Epoch 2/500
169/169 [==============================] - 2s 15ms/step - loss: 0.4402 - val_loss: 1.2464
Epoch 3/500
169/169 [==============================] - 3s 17ms/step - loss: 0.3348 - val_loss: 1.1791
Epoch 4/500
169/169 [==============================] - 2s 12ms/step - loss: 0.2703 - val_loss: 1.1696
Epoch 5/500
169/169 [==============================] - 2s 13ms/step - loss: 0.2257 - val_loss: 1.1793
Epoch 6/500
169/169 [==============================] - 2s 13ms/step - loss: 0.2013 - val_loss: 1.1050
Epoch 7/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1793 - val_loss: 1.1051
Epoch 8/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1640 - val_loss: 1.1824
Epoch 9/500
169/169 [==============================] - 3s 17ms/step - loss: 0.1520 - val_loss: 1.1328
Epoch 10/500
169/169 [==============================] - 3s 17ms/step - loss: 0.142

In [ ]:
#finger 2

#model.fit with early stopping
model.fit(R1_train, y_train_1[:,1], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model12.h5')

#predictions
test_pred12 = model.predict(R1_test)

Epoch 1/500
169/169 [==============================] - 3s 17ms/step - loss: 0.6180 - val_loss: 0.5703
Epoch 2/500
169/169 [==============================] - 3s 15ms/step - loss: 0.4674 - val_loss: 0.5222
Epoch 3/500
169/169 [==============================] - 2s 12ms/step - loss: 0.3507 - val_loss: 0.4591
Epoch 4/500
169/169 [==============================] - 2s 12ms/step - loss: 0.2705 - val_loss: 0.4585
Epoch 5/500
169/169 [==============================] - 2s 12ms/step - loss: 0.2207 - val_loss: 0.4623
Epoch 6/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1832 - val_loss: 0.4861
Epoch 7/500
169/169 [==============================] - 2s 13ms/step - loss: 0.1552 - val_loss: 0.4701
Epoch 8/500
169/169 [==============================] - 3s 16ms/step - loss: 0.1406 - val_loss: 0.4565
Epoch 9/500
169/169 [==============================] - 2s 13ms/step - loss: 0.1332 - val_loss: 0.4579
Epoch 10/500
169/169 [==============================] - 2s 12ms/step - loss: 0.116

In [ ]:
#finger 3

#model.fit with early stopping
model.fit(R1_train, y_train_1[:,2], epochs = 500, batch_size = 64, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model13.h5')

#predictions
test_pred13 = model.predict(R1_test)

Epoch 1/500
85/85 [==============================] - 2s 22ms/step - loss: 0.6145 - val_loss: 0.1606
Epoch 2/500
85/85 [==============================] - 2s 22ms/step - loss: 0.4667 - val_loss: 0.1660
Epoch 3/500
85/85 [==============================] - 2s 25ms/step - loss: 0.4152 - val_loss: 0.1635
Epoch 4/500
85/85 [==============================] - 2s 18ms/step - loss: 0.3731 - val_loss: 0.1703
Epoch 5/500
85/85 [==============================] - 1s 17ms/step - loss: 0.3247 - val_loss: 0.2052
Epoch 6/500
85/85 [==============================] - 1s 17ms/step - loss: 0.2856 - val_loss: 0.1972
Epoch 7/500
85/85 [==============================] - 2s 25ms/step - loss: 0.2499 - val_loss: 0.1946
Epoch 8/500
85/85 [==============================] - 2s 23ms/step - loss: 0.2248 - val_loss: 0.2001
Epoch 9/500
85/85 [==============================] - 1s 14ms/step - loss: 0.2048 - val_loss: 0.2242
Epoch 10/500
85/85 [==============================] - 2s 20ms/step - loss: 0.1859 - val_loss: 0.2297

In [ ]:
#finger 4

#model.fit with early stopping
model.fit(R1_train, y_train_1[:,3], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model14.h5')

#predictions
test_pred14 = model.predict(R1_test)

Epoch 1/500
169/169 [==============================] - 2s 14ms/step - loss: 0.5574 - val_loss: 0.6606
Epoch 2/500
169/169 [==============================] - 2s 14ms/step - loss: 0.3791 - val_loss: 0.6279
Epoch 3/500
169/169 [==============================] - 3s 17ms/step - loss: 0.2927 - val_loss: 0.6075
Epoch 4/500
169/169 [==============================] - 3s 16ms/step - loss: 0.2380 - val_loss: 0.6254
Epoch 5/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1919 - val_loss: 0.6100
Epoch 6/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1703 - val_loss: 0.6094
Epoch 7/500
169/169 [==============================] - 3s 21ms/step - loss: 0.1494 - val_loss: 0.6274
Epoch 8/500
169/169 [==============================] - 3s 19ms/step - loss: 0.1370 - val_loss: 0.6024
Epoch 9/500
169/169 [==============================] - 3s 17ms/step - loss: 0.1194 - val_loss: 0.6224
Epoch 10/500
169/169 [==============================] - 2s 14ms/step - loss: 0.110

In [ ]:
#finger 5

#model.fit with early stopping
model.fit(R1_train, y_train_1[:,4], epochs = 500, batch_size = 64, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model15.h5')

#predictions
test_pred15 = model.predict(R1_test)

Epoch 1/500
85/85 [==============================] - 1s 16ms/step - loss: 0.6762 - val_loss: 0.1476
Epoch 2/500
85/85 [==============================] - 1s 14ms/step - loss: 0.5636 - val_loss: 0.1599
Epoch 3/500
85/85 [==============================] - 1s 14ms/step - loss: 0.4231 - val_loss: 0.2545
Epoch 4/500
85/85 [==============================] - 1s 14ms/step - loss: 0.2954 - val_loss: 0.2740
Epoch 5/500
85/85 [==============================] - 1s 14ms/step - loss: 0.2310 - val_loss: 0.2467
Epoch 6/500
85/85 [==============================] - 1s 17ms/step - loss: 0.1886 - val_loss: 0.2923
Epoch 7/500
85/85 [==============================] - 1s 14ms/step - loss: 0.1642 - val_loss: 0.3188
Epoch 8/500
85/85 [==============================] - 2s 20ms/step - loss: 0.1446 - val_loss: 0.2882
Epoch 9/500
85/85 [==============================] - 2s 20ms/step - loss: 0.1315 - val_loss: 0.2924
Epoch 10/500
85/85 [==============================] - 2s 21ms/step - loss: 0.1118 - val_loss: 0.2824

In [ ]:
test_pred11 = test_pred11.reshape((-1,))
test_pred12 = test_pred12.reshape((-1,))
test_pred13 = test_pred13.reshape((-1,))
test_pred14 = test_pred14.reshape((-1,))
test_pred15 = test_pred15.reshape((-1,))

test_pred1 = np.vstack((test_pred11, test_pred12, test_pred13, test_pred14, test_pred15)).T

xs = np.linspace(0,test_pred1.shape[0],test_ecog1.shape[0])

#interpolation
y1 = np.empty_like(test_pred1[:,0])
cs1 = []
for i in range(test_pred1.shape[1]):
  x1 = np.arange(test_pred1.shape[0])
  y1 = test_pred1[:,i]
  cs1.append(CubicSpline(x1, y1, bc_type = 'clamped'))

interp_pred1_nn = np.vstack((cs1[0](xs),cs1[1](xs), cs1[2](xs), cs1[3](xs), cs1[4](xs))).T
interp_pred1_nn = apply_medfilt_to_predictions(interp_pred1_nn)

### Subject 2

The same model structure is reused for a second subject, allowing performance differences to reflect signal quality and subject variability rather than changes in code.

In [ ]:
#-----------Subject 2-------------

#scaling (normalizing) the data
scaler2 = StandardScaler()
R2_train = scaler2.fit_transform(R2_train)

scaler_filename = "scaler2.save"
joblib.dump(scaler2, scaler_filename)

R2_test = scaler2.transform(R2_test)

In [ ]:
#creating the neural network architecture
model = Sequential()
model.add(Dense(128, input_dim = R2_train.shape[1], activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(1, activation = 'linear'))

#choosing optimizer and loss
model.compile(optimizer = 'adam', loss = 'mean_absolute_error')

#early stopping
early_stop = EarlyStopping(monitor = 'val_loss', mode = 'min', verbose=1, patience = 25)

In [ ]:
#finger 1

#model.fit with early stopping
model.fit(R2_train, y_train_2[:,0], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model21.h5')

#predictions
test_pred21 = model.predict(R2_test)

Epoch 1/500
169/169 [==============================] - 4s 16ms/step - loss: 0.4830 - val_loss: 0.3881
Epoch 2/500
169/169 [==============================] - 2s 13ms/step - loss: 0.3861 - val_loss: 0.4084
Epoch 3/500
169/169 [==============================] - 2s 9ms/step - loss: 0.3259 - val_loss: 0.4289
Epoch 4/500
169/169 [==============================] - 2s 10ms/step - loss: 0.2612 - val_loss: 0.4202
Epoch 5/500
169/169 [==============================] - 2s 10ms/step - loss: 0.2269 - val_loss: 0.3827
Epoch 6/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1919 - val_loss: 0.4053
Epoch 7/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1715 - val_loss: 0.4179
Epoch 8/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1540 - val_loss: 0.3979
Epoch 9/500
169/169 [==============================] - 2s 15ms/step - loss: 0.1403 - val_loss: 0.4286
Epoch 10/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1261

In [ ]:
#finger 2

#model.fit with early stopping
model.fit(R2_train, y_train_2[:,1], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model22.h5')

#predictions
test_pred22 = model.predict(R2_test)

Epoch 1/500
169/169 [==============================] - 2s 14ms/step - loss: 0.5371 - val_loss: 0.7365
Epoch 2/500
169/169 [==============================] - 3s 15ms/step - loss: 0.4141 - val_loss: 0.9068
Epoch 3/500
169/169 [==============================] - 2s 11ms/step - loss: 0.3483 - val_loss: 0.8050
Epoch 4/500
169/169 [==============================] - 2s 10ms/step - loss: 0.2829 - val_loss: 0.8179
Epoch 5/500
169/169 [==============================] - 2s 10ms/step - loss: 0.2294 - val_loss: 0.8513
Epoch 6/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1989 - val_loss: 0.8672
Epoch 7/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1732 - val_loss: 0.7947
Epoch 8/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1472 - val_loss: 0.8365
Epoch 9/500
169/169 [==============================] - 2s 14ms/step - loss: 0.1344 - val_loss: 0.8718
Epoch 10/500
169/169 [==============================] - 3s 15ms/step - loss: 0.121

In [ ]:
#finger 3

#model.fit with early stopping
model.fit(R2_train, y_train_2[:,2], epochs = 500, batch_size = 64, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model23.h5')

#predictions
test_pred23 = model.predict(R2_test)

Epoch 1/500
85/85 [==============================] - 1s 13ms/step - loss: 0.6263 - val_loss: 0.2869
Epoch 2/500
85/85 [==============================] - 1s 12ms/step - loss: 0.5148 - val_loss: 0.3261
Epoch 3/500
85/85 [==============================] - 1s 12ms/step - loss: 0.4409 - val_loss: 0.4811
Epoch 4/500
85/85 [==============================] - 1s 12ms/step - loss: 0.3564 - val_loss: 0.3831
Epoch 5/500
85/85 [==============================] - 1s 12ms/step - loss: 0.2731 - val_loss: 0.5102
Epoch 6/500
85/85 [==============================] - 1s 12ms/step - loss: 0.2248 - val_loss: 0.5360
Epoch 7/500
85/85 [==============================] - 1s 12ms/step - loss: 0.1938 - val_loss: 0.5341
Epoch 8/500
85/85 [==============================] - 1s 12ms/step - loss: 0.1608 - val_loss: 0.5496
Epoch 9/500
85/85 [==============================] - 2s 18ms/step - loss: 0.1421 - val_loss: 0.6213
Epoch 10/500
85/85 [==============================] - 2s 18ms/step - loss: 0.1233 - val_loss: 0.6036

In [ ]:
#finger 4

#model.fit with early stopping
model.fit(R2_train, y_train_2[:,3], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model24.h5')

#predictions
test_pred24 = model.predict(R2_test)

Epoch 1/500
169/169 [==============================] - 2s 11ms/step - loss: 0.4139 - val_loss: 0.7290
Epoch 2/500
169/169 [==============================] - 2s 10ms/step - loss: 0.2975 - val_loss: 0.7112
Epoch 3/500
169/169 [==============================] - 2s 10ms/step - loss: 0.2226 - val_loss: 0.7089
Epoch 4/500
169/169 [==============================] - 2s 13ms/step - loss: 0.1817 - val_loss: 0.6965
Epoch 5/500
169/169 [==============================] - 3s 15ms/step - loss: 0.1508 - val_loss: 0.6891
Epoch 6/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1317 - val_loss: 0.6907
Epoch 7/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1186 - val_loss: 0.6556
Epoch 8/500
169/169 [==============================] - 2s 10ms/step - loss: 0.1071 - val_loss: 0.6671
Epoch 9/500
169/169 [==============================] - 2s 10ms/step - loss: 0.0972 - val_loss: 0.6795
Epoch 10/500
169/169 [==============================] - 2s 10ms/step - loss: 0.090

In [ ]:
#finger 5

#model.fit with early stopping
model.fit(R2_train, y_train_2[:,4], epochs = 500, batch_size = 64, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model25.h5')

#predictions
test_pred25 = model.predict(R2_test)

Epoch 1/500
85/85 [==============================] - 1s 13ms/step - loss: 0.4336 - val_loss: 0.7530
Epoch 2/500
85/85 [==============================] - 1s 12ms/step - loss: 0.3173 - val_loss: 0.7366
Epoch 3/500
85/85 [==============================] - 1s 12ms/step - loss: 0.2646 - val_loss: 0.7466
Epoch 4/500
85/85 [==============================] - 1s 12ms/step - loss: 0.1872 - val_loss: 0.7444
Epoch 5/500
85/85 [==============================] - 1s 14ms/step - loss: 0.1433 - val_loss: 0.7494
Epoch 6/500
85/85 [==============================] - 2s 18ms/step - loss: 0.1218 - val_loss: 0.7504
Epoch 7/500
85/85 [==============================] - 2s 19ms/step - loss: 0.1030 - val_loss: 0.7672
Epoch 8/500
85/85 [==============================] - 1s 16ms/step - loss: 0.0914 - val_loss: 0.7557
Epoch 9/500
85/85 [==============================] - 1s 12ms/step - loss: 0.0810 - val_loss: 0.7535
Epoch 10/500
85/85 [==============================] - 1s 12ms/step - loss: 0.0727 - val_loss: 0.7426

In [ ]:
test_pred21 = test_pred21.reshape((-1,))
test_pred22 = test_pred22.reshape((-1,))
test_pred23 = test_pred23.reshape((-1,))
test_pred24 = test_pred24.reshape((-1,))
test_pred25 = test_pred25.reshape((-1,))

test_pred2 = np.vstack((test_pred21, test_pred22, test_pred23, test_pred24, test_pred25)).T

#interpolation
y2 = np.empty_like(test_pred2[:,0])
cs2 = []
for i in range(test_pred2.shape[1]):
  x2 = np.arange(test_pred2.shape[0])
  y2 = test_pred2[:,i]
  cs2.append(CubicSpline(x2, y2, bc_type = 'clamped'))

interp_pred2_nn = np.vstack((cs2[0](xs),cs2[1](xs), cs2[2](xs), cs2[3](xs), cs2[4](xs))).T
interp_pred2_nn = apply_medfilt_to_predictions(interp_pred2_nn)

### Subject 3

The third subject provides another check on whether the decoding approach generalizes across separate ECoG recordings.

In [ ]:
#-----------Subject 3-------------

#scaling (normalizing) the data
scaler3 = StandardScaler()
R3_train = scaler3.fit_transform(R3_train)

scaler_filename = "scaler3.save"
joblib.dump(scaler3, scaler_filename)

R3_test = scaler3.transform(R3_test)

In [ ]:
#creating the neural network architecture
model = Sequential()
model.add(Dense(128, input_dim = R3_train.shape[1], activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(256, activation = 'relu'))
model.add(Dense(1, activation = 'linear'))

#choosing optimizer and loss
model.compile(optimizer = 'adam', loss = 'mean_absolute_error')

#early stopping
early_stop = EarlyStopping(monitor = 'val_loss', mode = 'min', verbose=1, patience = 25)

In [ ]:
#finger 1

#model.fit with early stopping
model.fit(R3_train, y_train_3[:,0], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model31.h5')

#predictions
test_pred31 = model.predict(R3_test)

Epoch 1/500
169/169 [==============================] - 5s 20ms/step - loss: 0.5872 - val_loss: 0.3221
Epoch 2/500
169/169 [==============================] - 2s 12ms/step - loss: 0.3612 - val_loss: 0.2947
Epoch 3/500
169/169 [==============================] - 2s 11ms/step - loss: 0.2547 - val_loss: 0.3204
Epoch 4/500
169/169 [==============================] - 2s 11ms/step - loss: 0.2015 - val_loss: 0.3126
Epoch 5/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1750 - val_loss: 0.2861
Epoch 6/500
169/169 [==============================] - 2s 13ms/step - loss: 0.1447 - val_loss: 0.3389
Epoch 7/500
169/169 [==============================] - 3s 17ms/step - loss: 0.1351 - val_loss: 0.3089
Epoch 8/500
169/169 [==============================] - 3s 15ms/step - loss: 0.1236 - val_loss: 0.3035
Epoch 9/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1146 - val_loss: 0.2882
Epoch 10/500
169/169 [==============================] - 2s 12ms/step - loss: 0.105

In [ ]:
#finger 2

#model.fit with early stopping
model.fit(R3_train, y_train_3[:,1], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model32.h5')

#predictions
test_pred32 = model.predict(R3_test)

Epoch 1/500
169/169 [==============================] - 2s 12ms/step - loss: 0.6786 - val_loss: 0.4594
Epoch 2/500
169/169 [==============================] - 2s 12ms/step - loss: 0.3962 - val_loss: 0.4062
Epoch 3/500
169/169 [==============================] - 2s 11ms/step - loss: 0.2831 - val_loss: 0.4344
Epoch 4/500
169/169 [==============================] - 3s 16ms/step - loss: 0.2210 - val_loss: 0.4174
Epoch 5/500
169/169 [==============================] - 3s 17ms/step - loss: 0.1875 - val_loss: 0.4152
Epoch 6/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1560 - val_loss: 0.4106
Epoch 7/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1365 - val_loss: 0.4257
Epoch 8/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1200 - val_loss: 0.4304
Epoch 9/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1098 - val_loss: 0.4303
Epoch 10/500
169/169 [==============================] - 2s 12ms/step - loss: 0.102

In [ ]:
#finger 3

#model.fit with early stopping
model.fit(R3_train, y_train_3[:,2], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model33.h5')

#predictions
test_pred33 = model.predict(R3_test)

Epoch 1/500
169/169 [==============================] - 2s 12ms/step - loss: 0.4731 - val_loss: 0.6644
Epoch 2/500
169/169 [==============================] - 2s 12ms/step - loss: 0.3063 - val_loss: 0.6504
Epoch 3/500
169/169 [==============================] - 2s 11ms/step - loss: 0.2253 - val_loss: 0.6184
Epoch 4/500
169/169 [==============================] - 2s 14ms/step - loss: 0.1786 - val_loss: 0.6773
Epoch 5/500
169/169 [==============================] - 3s 17ms/step - loss: 0.1482 - val_loss: 0.6319
Epoch 6/500
169/169 [==============================] - 2s 13ms/step - loss: 0.1285 - val_loss: 0.6597
Epoch 7/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1107 - val_loss: 0.6507
Epoch 8/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1008 - val_loss: 0.6385
Epoch 9/500
169/169 [==============================] - 2s 11ms/step - loss: 0.0922 - val_loss: 0.6508
Epoch 10/500
169/169 [==============================] - 2s 12ms/step - loss: 0.085

In [ ]:
#finger 4

#model.fit with early stopping
model.fit(R3_train, y_train_3[:,3], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model34.h5')

#predictions
test_pred34 = model.predict(R3_test)

Epoch 1/500
169/169 [==============================] - 3s 16ms/step - loss: 0.4972 - val_loss: 0.5471
Epoch 2/500
169/169 [==============================] - 3s 18ms/step - loss: 0.2599 - val_loss: 0.5814
Epoch 3/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1895 - val_loss: 0.5758
Epoch 4/500
169/169 [==============================] - 3s 16ms/step - loss: 0.1504 - val_loss: 0.6071
Epoch 5/500
169/169 [==============================] - 3s 16ms/step - loss: 0.1274 - val_loss: 0.5541
Epoch 6/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1062 - val_loss: 0.5681
Epoch 7/500
169/169 [==============================] - 2s 12ms/step - loss: 0.0964 - val_loss: 0.5463
Epoch 8/500
169/169 [==============================] - 2s 12ms/step - loss: 0.0888 - val_loss: 0.5626
Epoch 9/500
169/169 [==============================] - 2s 11ms/step - loss: 0.0845 - val_loss: 0.5443
Epoch 10/500
169/169 [==============================] - 2s 12ms/step - loss: 0.079

In [ ]:
#finger 5

#model.fit with early stopping
model.fit(R3_train, y_train_3[:,4], epochs = 500, batch_size = 32, verbose = 1, validation_split = 0.1, callbacks = [early_stop])
model.save('model35.h5')

#predictions
test_pred35 = model.predict(R3_test)

Epoch 1/500
169/169 [==============================] - 3s 17ms/step - loss: 0.3678 - val_loss: 0.4859
Epoch 2/500
169/169 [==============================] - 2s 14ms/step - loss: 0.1967 - val_loss: 0.3245
Epoch 3/500
169/169 [==============================] - 2s 11ms/step - loss: 0.1322 - val_loss: 0.4453
Epoch 4/500
169/169 [==============================] - 2s 12ms/step - loss: 0.1012 - val_loss: 0.4005
Epoch 5/500
169/169 [==============================] - 2s 11ms/step - loss: 0.0818 - val_loss: 0.4652
Epoch 6/500
169/169 [==============================] - 2s 11ms/step - loss: 0.0724 - val_loss: 0.4859
Epoch 7/500
169/169 [==============================] - 2s 14ms/step - loss: 0.0646 - val_loss: 0.4158
Epoch 8/500
169/169 [==============================] - 3s 17ms/step - loss: 0.0602 - val_loss: 0.4057
Epoch 9/500
169/169 [==============================] - 2s 13ms/step - loss: 0.0546 - val_loss: 0.5168
Epoch 10/500
169/169 [==============================] - 2s 11ms/step - loss: 0.051

In [ ]:
test_pred31 = test_pred31.reshape((-1,))
test_pred32 = test_pred32.reshape((-1,))
test_pred33 = test_pred33.reshape((-1,))
test_pred34 = test_pred34.reshape((-1,))
test_pred35 = test_pred35.reshape((-1,))

test_pred3 = np.vstack((test_pred31, test_pred32, test_pred33, test_pred34, test_pred35)).T

#interpolation
y3 = np.empty_like(test_pred3[:,0])
cs3 = []
for i in range(test_pred3.shape[1]):
  x3 = np.arange(test_pred3.shape[0])
  y3 = test_pred3[:,i]
  cs3.append(CubicSpline(x3, y3, bc_type = 'clamped'))

interp_pred3_nn = np.vstack((cs3[0](xs),cs3[1](xs), cs3[2](xs), cs3[3](xs), cs3[4](xs))).T
interp_pred3_nn = apply_medfilt_to_predictions(interp_pred3_nn)

## Prediction Stacking

Predictions from all subjects are combined into a single output matrix for downstream scoring or downstream evaluation.

In [ ]:
#-------STACKING ALL PREDICTIONS-----------
predictions = np.zeros((3,1), dtype=object)
predictions[0,0] = interp_pred1_nn
predictions[1,0] = interp_pred2_nn
predictions[2,0] = interp_pred3_nn

savemat('predictions.mat', {'predicted_dg':predictions})